In [ ]:
import boto3
import pandas as pd
import io
from sklearn.preprocessing import StandardScaler

In [ ]:
bucket = 'cust-seg-ygp'
rfm_key = 'rfm_table.csv'        # file you previously saved
rfm_scaled_key = 'rfm_scaled.csv'  # output file name to save

s3 = boto3.client('s3')
obj = s3.get_object(Bucket=bucket, Key=rfm_key)
rfm = pd.read_csv(obj['Body'])
print("Loaded RFM shape:", rfm.shape)
display(rfm.head())

Loaded RFM shape: (5878, 4)


In [ ]:
# 4) pick columns to scale (DO NOT scale Customer ID)
features = ['Recency', 'Frequency', 'Monetary']
rfm_features = rfm[features]

In [ ]:
# check if there are any missing values
print("Any missing values in features?\n", rfm_features.isna().sum())

Any missing values in features?
 Recency      0
Frequency    0
Monetary     0
dtype: int64


In [ ]:
# 5) scale using StandardScaler
scaler = StandardScaler()
scaled_values = scaler.fit_transform(rfm_features)   # result is numpy array

In [ ]:
# 6) convert scaled values back to dataframe with readable column names
rfm_scaled = pd.DataFrame(scaled_values, columns=['Recency_scaled', 'Frequency_scaled', 'Monetary_scaled'])

In [ ]:
# 7) combine scaled columns with Customer ID so each row is still traceable
rfm_out = pd.concat([rfm[['Customer ID']].reset_index(drop=True), rfm_scaled.reset_index(drop=True)], axis=1)
print("Scaled RFM preview:")
display(rfm_out.head())

Scaled RFM preview:


In [ ]:
# 8) save scaled rfm to S3 
csv_buffer = io.StringIO()
rfm_out.to_csv(csv_buffer, index=False)
s3.put_object(Bucket=bucket, Key=rfm_scaled_key, Body=csv_buffer.getvalue())
print(f"Saved scaled RFM to s3://{bucket}/{rfm_scaled_key}")

Saved scaled RFM to s3://cust-seg-ygp/rfm_scaled.csv


In [ ]:
rfm_scaled.describe()